# MovieLens — Analysis

Data preprocessing is done in `preprocessing.ipynb`.



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np    

In [ ]:
# Load preprocessed data (run preprocessing.ipynb first if needed)
df = pd.read_csv("../data/merged_cleaned.csv")

print(f"Loaded df dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Define column names for analysis
HUMAN_COLS = [c for c in df.columns if c.startswith("human_")]
LLM_COLS = ["GPT-4o-mini"]
MIN, MAX = 1, 5

print(f"Human columns: {HUMAN_COLS}")
print(f"LLM columns: {LLM_COLS}")
print(f"Score range: [{MIN}, {MAX}]")

## Analysis

In [ ]:
# Fix import path - add parent directory to Python path
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..', '..'))

from core.llm_good_enough import LLMGoodEnough

In [ ]:
good_enough = LLMGoodEnough(
    df=df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=MIN, max_score=MAX
)

good_enough.visualize_good_enough(llm_col="GPT-4o-mini", y_lim=0.6)

In [ ]:
human_human_disagreements = good_enough.compute_human_disagreements()

print(human_human_disagreements.shape)
# mean and std (=2 decimal places)
print(f"Mean: {round(human_human_disagreements.mean(), 2)}")
print(f"Std: {round(human_human_disagreements.std(), 2)}")

In [ ]:
# Re-run analysis without dropna (using full df dataset)
good_enough = LLMGoodEnough(
    df=df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=MIN, max_score=MAX
)

good_enough.visualize_good_enough(llm_col="GPT-4o-mini", y_lim=0.6)

In [ ]:
good_enough = LLMGoodEnough(
    df=df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=1, max_score=5
)

good_enough.plot_judges_grid(
    y_lim=0.8,
    save_path="../svg/movielens_barplot.svg"
)

## Robustness (Monte Carlo)

In [ ]:
good_enough = LLMGoodEnough(
    df=df,
    human_cols=HUMAN_COLS,
    llm_cols=LLM_COLS,
    min_score=MIN,
    max_score=MAX,
    verbosity=0
)

good_enough.plot_monte_carlo_robustness(
    llm_col="GPT-4o-mini",
    save_path="../svg/monte_carlo_robustness.svg"
)

In [ ]:
good_enough.plot_human_stability_analysis(
    convergence_threshold=0.000005,
    min_iterations=20_000,
    max_iterations=500_000,
    parallel=True,
    save_path="../svg/human_stability_analysis.svg"
) 